# Diabetes Prediction Using Ensemble Learning

**Comparison:** Random Forest vs Gradient Boosting  
**Dataset:** Kaggle `iammustafatz/diabetes-prediction-dataset`

## Project objective
Predict the binary `diabetes` outcome and compare two ensemble classifiers. This notebook covers the faculty rubric: dataset understanding, preprocessing, model application, randomized tuning, evaluation, visualization, manual metric calculation, comparison, and interpretation.

> This is an educational model study, not a clinical diagnosis.

## 1. Imports and configuration

The data file is a variable so the exact faculty CSV can be used without changing the methodology. The reusable training functions keep this notebook and the deployed API consistent.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.train_models import (
    FEATURES, TARGET, build_pipeline, clean_data, evaluate, load_data,
    resolve_data_file, save_visualizations,
)

DATA_FILE = ROOT / 'data' / 'diabetes_prediction_dataset.csv'
if not DATA_FILE.exists():
    DATA_FILE = resolve_data_file(None)
if DATA_FILE is None:
    raise FileNotFoundError('Add the faculty CSV to data/ or update DATA_FILE.')
print(f'Using: {DATA_FILE}')

## 2. Dataset understanding

The dataset contains demographic, lifestyle, and clinical measurements. The target is `diabetes`: 0 means no recorded diabetes and 1 indicates diabetes. The next cell reports record count, data types, missing values, target balance, and a sample from the supplied file.

In [ ]:
raw_df, source = load_data(DATA_FILE, '')
df = clean_data(raw_df)
print('Source:', source)
print('Shape:', df.shape)
display(df.head())
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique()}))
display(df[TARGET].value_counts().rename('records').to_frame())

## 3. Preprocessing decisions

1. Numeric values are converted to numeric types and missing values use the training median.
2. Categorical values use the most frequent category and one-hot encoding.
3. Numeric fields are standardized inside the pipeline for a reproducible data contract.
4. IQR clipping reduces extreme data-entry influence; limits are learned only on training data.
5. Stratification keeps both target classes represented in the split and cross-validation.

In [ ]:
numeric_columns = ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level']
categorical_columns = ['gender', 'smoking_history']
print('Numeric features:', numeric_columns)
print('Categorical features:', categorical_columns)
display(df[FEATURES].isna().sum().to_frame('missing before pipeline imputation'))

## 4. Model application and parameter tuning

Random Forest uses many independent trees and averages their votes. Gradient Boosting grows trees sequentially; each new tree focuses on the current errors. `RandomizedSearchCV` samples model-specific hyperparameters and scores each configuration with ROC-AUC.

In [ ]:
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_spaces = {
    'random_forest': (
        build_pipeline(RandomForestClassifier(random_state=42, n_jobs=-1)),
        {'model__n_estimators': [150, 250, 350], 'model__max_depth': [None, 8, 12, 18],
         'model__min_samples_leaf': [1, 2, 4, 8], 'model__max_features': ['sqrt', 'log2', 0.7],
         'model__class_weight': [None, 'balanced', 'balanced_subsample']},
    ),
    'gradient_boosting': (
        build_pipeline(GradientBoostingClassifier(random_state=42)),
        {'model__n_estimators': [100, 150, 250], 'model__learning_rate': [0.03, 0.05, 0.1, 0.15],
         'model__max_depth': [2, 3, 4], 'model__min_samples_leaf': [1, 3, 6], 'model__subsample': [0.8, 1.0]},
    ),
}

searches = {}
evaluations = {}
for name, (pipeline, params) in search_spaces.items():
    search = RandomizedSearchCV(pipeline, params, n_iter=8, scoring='roc_auc', cv=cv, n_jobs=-1, random_state=42, refit=True)
    search.fit(X_train, y_train)
    searches[name] = search
    evaluations[name] = evaluate(name, search, X_test, y_test)
    print(name, evaluations[name]['best_params'])

## 5. Evaluation metrics

Accuracy measures total correctness. Precision asks how many predicted positives are truly positive. Recall asks how many actual positives were detected. F1 balances precision and recall. ROC-AUC summarizes probability ranking quality. The confusion matrix makes false positives and false negatives visible.

In [ ]:
metrics_table = pd.DataFrame({name: result['metrics'] for name, result in evaluations.items()}).T
display(metrics_table.style.format('{:.3f}'))
for name, result in evaluations.items():
    print(f'\n{name.replace("_", " ").title()} manual metrics:')
    print(result['manual_metrics'])
    ConfusionMatrixDisplay.from_predictions(y_test, result['predictions'], cmap='Blues')
    plt.title(name.replace('_', ' ').title())
    plt.show()

### Manual calculation example

For a binary confusion matrix with TN, FP, FN, and TP:

- `accuracy = (TP + TN) / (TP + TN + FP + FN)`
- `precision = TP / (TP + FP)`
- `recall = TP / (TP + FN)`

The next cell performs this calculation independently of scikit-learn's convenience functions.

In [ ]:
rf_predictions = evaluations['random_forest']['predictions']
tn, fp, fn, tp = confusion_matrix(y_test, rf_predictions, labels=[0, 1]).ravel()
total = tn + fp + fn + tp
manual_accuracy = (tp + tn) / total
manual_precision = tp / (tp + fp) if (tp + fp) else 0
manual_recall = tp / (tp + fn) if (tp + fn) else 0
print({'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp})
print({'accuracy': manual_accuracy, 'precision': manual_precision, 'recall': manual_recall})

## 6. Customized visualizations and feature interpretation

The plots are created from this dataset and the held-out predictions: target balance, target correlations, confusion matrices, ROC curves, and metric comparison. Feature importance shows which transformed signals each ensemble used most heavily. It indicates model reliance, not causation.

In [ ]:
save_visualizations(df, y_test, evaluations)
for name, result in evaluations.items():
    print(name, result['feature_importance'][:8])
print('Saved customized plots to outputs/')

## 7. Comparison and conclusion

Compare the two rows in the metric table rather than relying on accuracy alone. If Gradient Boosting leads on ROC-AUC/F1, sequential error-correction may be capturing useful interactions. If Random Forest is close or better, averaging may be generalizing more stably for this dataset.

For the final report, state the actual winning model, quote the actual values, discuss the precision–recall trade-off, and mention that a benchmark probability is not a diagnosis.

### Rubric checklist
- Dataset features, target, records, missing values, and data types: sections 1–2
- Preprocessing and justification: section 3
- Algorithm steps and application: section 4
- Randomized parameter tuning: section 4
- Metrics and manual calculation: section 5
- Comparison, visuals, interpretation, and conclusion: sections 5–7